# Error Handling and Validation

Programs fail in predictable ways: bad input, missing files, unexpected values, and invalid assumptions.
Good software does not pretend errors never happen. It handles them deliberately.

## Learning goals

By the end of this section, you should be able to:

- explain the difference between validation and exception handling
- use `try`, `catch`, and `finally` correctly
- prefer `TryParse` over `Parse` for user input
- write clear, actionable error messages
- decide when to recover and when to stop execution

## Validation vs. exceptions

- **Validation** checks expected bad input before doing risky work.
- **Exceptions** handle unexpected failures at runtime.

Validation should be your first line of defense. Exceptions are a fallback mechanism when something still goes wrong.

## Basic `try/catch/finally`

```csharp
try
{
    // Code that may fail at runtime
    int value = int.Parse(Console.ReadLine()!);
    Console.WriteLine($"You entered {value}");
}
catch (FormatException)
{
    Console.WriteLine("Input must be a whole number.");
}
catch (OverflowException)
{
    Console.WriteLine("Number is outside the valid int range.");
}
finally
{
    // Runs whether an exception occurred or not
    Console.WriteLine("Input attempt complete.");
}
```

Use `finally` for cleanup (closing resources, resetting state, releasing handles).

## Prefer `TryParse` for user input

If invalid input is expected, avoid exceptions entirely:

```csharp
Console.Write("Enter age: ");
string raw = Console.ReadLine() ?? "";

if (int.TryParse(raw, out int age) && age >= 0)
{
    Console.WriteLine($"Age recorded: {age}");
}
else
{
    Console.WriteLine("Please enter a non-negative whole number.");
}
```

Guideline:

- use `TryParse` for normal input flow
- use `try/catch` around operations that can fail unpredictably (I/O, network, external dependencies)

## File I/O example with safe handling

```csharp
string path = "scores.txt";

try
{
    using StreamReader reader = new StreamReader(path);
    string? line;
    while ((line = reader.ReadLine()) != null)
    {
        Console.WriteLine(line);
    }
}
catch (FileNotFoundException)
{
    Console.WriteLine($"Could not find file: {path}");
}
catch (UnauthorizedAccessException)
{
    Console.WriteLine("Permission denied while reading the file.");
}
catch (IOException ex)
{
    Console.WriteLine($"I/O error: {ex.Message}");
}
```

Catch specific exception types before broad ones.

## Throwing exceptions intentionally

Throw when an API is used incorrectly and execution should stop:

```csharp
public static double Divide(double numerator, double denominator)
{
    if (denominator == 0)
        throw new ArgumentException("Denominator cannot be zero.", nameof(denominator));

    return numerator / denominator;
}
```

This is better than returning meaningless values that hide bugs.

## Error message quality

Bad message:

- `"Error happened"`

Better message:

- `"Could not open 'scores.txt'. Check that the file exists and that you have read permission."`

Good messages are:

- specific
- user-actionable
- free of internal noise unless debugging

## Common mistakes

- catching `Exception` everywhere and ignoring details
- swallowing exceptions without logging or reporting
- using exceptions for normal control flow
- showing stack traces directly to end users

## Quick practice

1. Read a value from the console and keep prompting until a valid `double` is entered.
2. Open a file path provided by the user and print a friendly message for:
   - file not found
   - access denied
   - unknown I/O error
3. Add argument checks to one of your existing methods and throw `ArgumentException` when input is invalid.

## Summary

Robust code assumes failures will occur and handles them clearly:

- validate expected input first
- catch only what you can handle
- clean up resources predictably
- fail loudly when assumptions are violated
